In [ ]:
import os
import re
from collections import defaultdict, Counter
from datetime import datetime
from netCDF4 import Dataset

# ============ 配置区 ============
target_dirs = [
    "/mnt/backup_ETH/EXTR_2000/B2000WCN.NOCOUPL.e122.f19_g16.010/atm/hist",
    "/mnt/backup_ETH/EXTR_2000/B2000WCN.NOCOUPL.e122.f19_g16.011/atm/hist",
]
out_root = "/home/weiji/restart_exam"
# =================================

CAM_STREAM_RE = re.compile(r"\.cam\.(h\d)\.")
IS_EXTR_RE    = re.compile(r"\.extr(\.|$)", re.IGNORECASE)

def classify_file(basename: str):
    m = CAM_STREAM_RE.search(basename)
    if not m or ".cam." not in basename or not basename.endswith(".nc"):
        return None
    stream = m.group(1)
    is_extr = bool(IS_EXTR_RE.search(basename))
    case_id = basename.split(".cam.")[0]
    class_key = f"{case_id}.cam.{stream}" + (" [EXTR]" if is_extr else "")
    return class_key, stream, is_extr

def read_time_len_and_meta(path: str, want_meta=False):
    """只读必要信息；默认仅 time 维长度。"""
    out = {"ok": False, "error": None, "time_len": None}
    if want_meta:
        out.update({
            "dims_sig": None,
            "lat_n": None, "lon_n": None, "lev_n": None, "ilev_n": None,
            "calendar": None, "time_units": None,
            "globals": None,
        })
    try:
        d = Dataset(path, "r")
        if "time" in d.dimensions:
            out["time_len"] = int(d.dimensions["time"].size)
        elif "time" in d.variables:
            out["time_len"] = int(d.variables["time"].shape[0])
        else:
            out["time_len"] = 0

        if want_meta:
            dims_sig = tuple(sorted((k, int(v.size)) for k, v in d.dimensions.items()))
            out["dims_sig"] = dims_sig
            out["lat_n"]  = int(d.dimensions["lat"].size)  if "lat"  in d.dimensions else None
            out["lon_n"]  = int(d.dimensions["lon"].size)  if "lon"  in d.dimensions else None
            out["lev_n"]  = int(d.dimensions["lev"].size)  if "lev"  in d.dimensions else None
            out["ilev_n"] = int(d.dimensions["ilev"].size) if "ilev" in d.dimensions else None
            if "time" in d.variables:
                v = d.variables["time"]
                out["calendar"]   = getattr(v, "calendar", None)
                out["time_units"] = getattr(v, "units", None)
            g = {}
            for k in ("case","source","title","initial_file","history"):
                if k in d.ncattrs():
                    g[k] = str(getattr(d, k))[:800]
            out["globals"] = g
        d.close()
        out["ok"] = True
        return out
    except Exception as e:
        out["error"] = str(e)
        return out

def pick_samples(lst):
    """取首、中、末三个样本"""
    if not lst: return []
    idxs = {0, len(lst)-1, len(lst)//2}
    return [lst[i] for i in sorted(idxs)]

# ============ 主逻辑 ============
for target_dir in target_dirs:
    if not os.path.isdir(target_dir):
        print(f"❌ 目录不存在：{target_dir}")
        continue

    case_name = os.path.basename(os.path.dirname(target_dir))
    out_txt = os.path.join(out_root, f"{case_name}_atm_hist_fast.txt")

    files = [f for f in sorted(os.listdir(target_dir))
             if f.endswith(".nc") and ".cam." in f]
    groups = defaultdict(list)
    for f in files:
        cls = classify_file(f)
        if cls:
            class_key, stream, is_extr = cls
            groups[class_key].append(os.path.join(target_dir, f))

    with open(out_txt, "w", encoding="utf-8") as out:
        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        out.write(f"# FAST SCAN: {target_dir}\n")
        out.write(f"# created: {now}\n\n")

        for class_key in sorted(groups.keys()):
            flist = groups[class_key]
            flist.sort()
            out.write(f"[TYPE] {class_key}\n")
            out.write(f"  - files.count   : {len(flist)}\n")
            out.write(f"  - first_file    : {os.path.basename(flist[0])}\n")
            out.write(f"  - last_file     : {os.path.basename(flist[-1])}\n")

            # 一个样例文件取基础元信息
            meta = read_time_len_and_meta(flist[0], want_meta=True)
            if meta["ok"]:
                out.write(f"  - grid(lat,lon) : {meta['lat_n']} x {meta['lon_n']}\n")
                out.write(f"  - lev/ilev      : {meta['lev_n']} / {meta['ilev_n']}\n")
                out.write(f"  - calendar      : {meta['calendar']}\n")
                if meta["globals"]:
                    for k,v in meta["globals"].items():
                        out.write(f"  - {k:13s}: {v}\n")
            else:
                out.write(f"  - meta.error    : {meta['error']}\n")

            # 抽样 3 个文件看 time_len
            samples = pick_samples(flist)
            tlens = []
            for p in samples:
                m = read_time_len_and_meta(p)
                tl = m["time_len"] if m["ok"] else "ERR"
                tlens.append((os.path.basename(p), tl))
            out.write(f"  - time_len.sample: {tlens}\n")

            # 若样本一致，直接汇总；否则全扫 time_len
            setlen = {t for _, t in tlens if isinstance(t, int)}
            if len(setlen) <= 1:
                only = list(setlen)[0] if setlen else None
                out.write(f"  - time_len.dist : {{ {only}:{len(flist)} }} (uniform)\n\n")
                continue

            dist = Counter()
            for p in flist:
                m = read_time_len_and_meta(p)
                if m["ok"]:
                    dist[m["time_len"]] += 1
            pretty = ", ".join(f"{k}:{v}" for k,v in sorted(dist.items()))
            out.write(f"  - time_len.dist : {pretty}\n\n")

    print(f"✅ 输出完成：{out_txt}")


In [ ]:
import os, re, glob
from collections import defaultdict, Counter
from datetime import datetime
from netCDF4 import Dataset
import numpy as np

ROOT = "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN"
OUT  = "/home/weiji/restart_exam/CO2x1SmidEmin_scan.txt"

# 识别 cam 流
CAM_STREAM_RE = re.compile(r"\.cam\.(h\d)\.")
IS_EXTR_RE    = re.compile(r"\.extr(\.|$)", re.IGNORECASE)

# 想要扫描的常见子目录（顺序会按存在与否过滤）
SUBDIR_GUESSES = [
    "atm/hist", "to-publish", "extract", "EPF", "polcap", "ICEFRAC", 
    "daily", "episodic", "derived", "postproc"
]

def human_size(n):
    for u in ["B","KB","MB","GB","TB"]:
        if n < 1024: return f"{n:.1f}{u}"
        n /= 1024
    return f"{n:.1f}PB"

def list_all_nc_files(root):
    out = []
    for dirpath, dirnames, filenames in os.walk(root):
        for fn in filenames:
            if fn.endswith(".nc"):
                out.append(os.path.join(dirpath, fn))
    return sorted(out)

def classify_kind(path):
    """
    输出 (group_key, kind_note)
    group_key 用于汇总；kind_note 用于描述
    """
    rel = os.path.relpath(path, ROOT)
    parts = rel.split(os.sep)
    # 粗分类：组件/常见专题目录
    if len(parts) >= 3 and parts[1] == "hist" and ".cam." in parts[-1]:
        # 典型 CAM 输出：按 h* + [EXTR] 分组
        base = os.path.basename(path)
        m = CAM_STREAM_RE.search(base)
        stream = m.group(1) if m else "h?"
        extr   = " [EXTR]" if IS_EXTR_RE.search(base) else ""
        return (f"atm/hist :: {stream}{extr}", "CAM history stream")
    # 其余：按二级目录名粗分
    if len(parts) >= 2:
        key = f"{parts[0]}/{parts[1]}"
    else:
        key = parts[0]
    return (key, "derived/postproc or specialized")

def read_header_quick(nc_path, want_meta=True):
    """
    只读头与少量内容（极轻量）：time维长度、网格、lev、日历/units、变量数、全局属性片段
    再尝试从一个变量上抽一个很小的切片做一下统计（均值），验证可读。
    """
    res = {
        "ok": False, "err": None,
        "dims": {}, "n_vars": 0,
        "lat": None, "lon": None, "lev": None, "ilev": None, "time": None,
        "calendar": None, "time_units": None,
        "globals": {}, "sample_var": None, "sample_stat": None, "shape": None,
    }
    try:
        d = Dataset(nc_path, "r")
        # 基本维度
        res["dims"] = {k: int(v.size) for k, v in d.dimensions.items()}
        res["n_vars"] = sum(1 for k in d.variables if k not in d.dimensions)
        for k in ("lat","lon","lev","ilev","time"):
            if k in d.dimensions:
                res[k] = int(d.dimensions[k].size)
        if "time" in d.variables:
            tv = d.variables["time"]
            res["calendar"]   = getattr(tv, "calendar", None)
            res["time_units"] = getattr(tv, "units", None)
        # 全局属性（挑重点）
        for k in ("case","source","title","initial_file","history","logname","host"):
            if k in d.ncattrs():
                v = str(getattr(d, k))
                res["globals"][k] = v[:400]  # 截断避免太长
        # 抽一个变量（优先常见变量；否则挑第一个数值型）
        prefer = ["U","V","T","O3","Q","Z3","PS","PSL","TREFHT","TS","QRL","QRS","CLDTOT"]
        cand = None
        for nm in prefer:
            if nm in d.variables:
                cand = nm; break
        if cand is None:
            # 挑一个数值型且至少2维的变量
            for nm, var in d.variables.items():
                if nm in d.dimensions: 
                    continue
                if hasattr(var, "dtype") and np.issubdtype(var.dtype, np.number):
                    # 尽量不要太高维；能切就行
                    cand = nm; break
        if cand is not None:
            var = d.variables[cand]
            # 构造一个非常小的切片（每个维度取前2个）
            slicer = []
            for s in var.shape:
                if s is None: 
                    slicer.append(slice(0,1))
                else:
                    slicer.append(slice(0, min(2, s)))
            arr = var[tuple(slicer)]
            # 计算简单统计
            with np.errstate(all='ignore'):
                m = float(np.nanmean(arr))
            res["sample_var"]  = cand
            res["sample_stat"] = f"nanmean={m:.6g}"
            res["shape"]       = f"{var.shape} (sliced -> {[min(2,s) for s in var.shape]})"
        d.close()
        res["ok"] = True
        return res
    except Exception as e:
        res["err"] = str(e)
        return res

# —— 扫描有哪些“数据类型”(group)；每类挑一个例子文件读取 ——
all_nc = list_all_nc_files(ROOT)
groups = defaultdict(list)
sizes  = {}
for p in all_nc:
    k, note = classify_kind(p)
    groups[(k, note)].append(p)
    try:
        sizes[p] = os.path.getsize(p)
    except:
        sizes[p] = 0

lines = []
lines.append(f"# SCAN of {ROOT}")
lines.append(f"# created: {datetime.now()}")
lines.append("")
lines.append("## 目录结构与数据类型（按组汇总）")
lines.append("")

for (k, note) in sorted(groups.keys(), key=lambda x: x[0]):
    flist = groups[(k, note)]
    total_sz = sum(sizes.get(p,0) for p in flist)
    lines.append(f"[GROUP] {k}   :: {note}")
    lines.append(f"  - files.count : {len(flist)}")
    lines.append(f"  - total.size  : {human_size(total_sz)}")
    # 取一个“代表文件”：优先较新的/较大的
    flist.sort()
    ex = max(flist, key=lambda p: (sizes.get(p,0), p))  # 简单策略
    lines.append(f"  - example     : {ex}")
    # 读取头信息 + 小片统计
    meta = read_header_quick(ex, want_meta=True)
    if meta["ok"]:
        dims_sig = ", ".join([f"{k}:{v}" for k,v in sorted(meta["dims"].items())])
        lines.append(f"  - dims        : {dims_sig}")
        grid = f"{meta['lat']}x{meta['lon']}" if meta["lat"] and meta["lon"] else f"lat={meta['lat']},lon={meta['lon']}"
        lines.append(f"  - grid        : {grid}, lev={meta['lev']}, ilev={meta['ilev']}, time={meta['time']}")
        lines.append(f"  - calendar    : {meta['calendar']} ; units: {meta['time_units']}")
        lines.append(f"  - n_vars      : {meta['n_vars']}")
        if meta["globals"]:
            gkeep = ["case","source","title","initial_file","history","logname","host"]
            for kk in gkeep:
                if kk in meta["globals"]:
                    lines.append(f"  - {kk:12s}: {meta['globals'][kk]}")
        if meta["sample_var"]:
            lines.append(f"  - sample.var  : {meta['sample_var']}")
            lines.append(f"  - sample.stat : {meta['sample_stat']}")
            lines.append(f"  - sample.shape: {meta['shape']}")
    else:
        lines.append(f"  - read.error  : {meta['err']}")
    lines.append("")

# 写出报告
os.makedirs(os.path.dirname(OUT), exist_ok=True)
with open(OUT, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f"✅ 已写出扫描与样例读取报告：{OUT}")
print("内容包括：各类数据的文件数量、体量、一个代表文件的完整路径、dims/网格/lev/日历/变量数、全局属性片段、一个小变量的轻量统计（均值）。")


In [ ]:
import os
from datetime import datetime

# ===== CONFIG =====
ROOT = "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN"
OUT_TXT = "/home/weiji/restart_exam/CO2x1SmidEmin_filetree.txt"
INCLUDE_ONLY_NC = False   # 若只想看 .nc 文件可改 True

# ===== EXECUTION =====
lines = []
lines.append(f"# FILE TREE for: {ROOT}")
lines.append(f"# Generated at: {datetime.now()}")
lines.append("")

for root, dirs, files in os.walk(ROOT):
    # 计算缩进层级
    level = root.replace(ROOT, "").count(os.sep)
    indent = "    " * level
    lines.append(f"{indent}[DIR] {root}")
    # 文件部分
    for f in sorted(files):
        if INCLUDE_ONLY_NC and not f.endswith(".nc"):
            continue
        subindent = "    " * (level + 1)
        fpath = os.path.join(root, f)
        try:
            size_mb = round(os.path.getsize(fpath)/1024/1024, 2)
        except:
            size_mb = "?"
        lines.append(f"{subindent}|-- {f} ({size_mb} MB)")
    lines.append("")

os.makedirs(os.path.dirname(OUT_TXT), exist_ok=True)
with open(OUT_TXT, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f"✅ 目录扫描完成：{OUT_TXT}")


In [ ]:
import os
import re
from pathlib import Path
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Optional

import xarray as xr
import numpy as np

ROOT = Path("/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN")
OUT  = Path("/home/weiji/restart_exam/CO2x1SmidEmin_summary.txt")

# ----------------------------
# 1) 定义“文件类型 -> 匹配规则”
# ----------------------------
PATTERNS = {
    # 月平均原始提取：...cam.h0.YYYY-MM.nc.extr.nc
    "cam.h0.monthly_extr": re.compile(r"\.cam\.h0\.\d{4}-\d{2}\.nc\.extr\.nc$"),

    # h0 段聚合/派生：...cam.h0.0101-0300.xxx.nc 或 ...cam.h0.0100-0500_T2Mz_U.nc 等
    "cam.h0.chunk": re.compile(r"\.cam\.h0\.\d{4}-\d{4}.*\.nc$"),

    # h1 单点等压快照（文件名显式变量 U；如果将来出现 T/V/Z3/O3 也可扩展）
    "cam.h1.isobar.tmp": re.compile(r"\.cam\.h1\.\d+\.([A-Za-z0-9]+)\.isobar\.tmp\.nc$"),

    # h1 聚合产品（含 T2Ms_* / O3Col / QRS.zm / ehflux.isobar 等）
    "cam.h1.chunk_products": re.compile(r"\.cam\.h1\.\d{4}-\d{4}.*\.(nc|zm\.nc)$"),

    # NAM 合并
    "NAM.merged": re.compile(r"\.0101-0300_NAM\.nc$")
}

# ----------------------------
# 2) 遍历目录，按类型归类
# ----------------------------
def collect_files(root: Path) -> Dict[str, List[Path]]:
    buckets: Dict[str, List[Path]] = defaultdict(list)
    for p in root.rglob("*.nc"):
        # 只记录我们关心的类型；优先级从更具体到更宽泛
        rel = str(p)
        matched = False
        for key, pat in PATTERNS.items():
            if pat.search(rel):
                buckets[key].append(p)
                matched = True
                break
        if not matched:
            # 其余 .nc 归为 OTHER，便于人工检查（比如 epflux 等）
            buckets["OTHER"].append(p)
    return buckets

# ----------------------------
# 3) 从文件读取 “时间范围/分辨率/变量名” (抽样 1~2 个)
# ----------------------------
def safe_read_meta(nc_path: Path) -> dict:
    info = {
        "time_min": None, "time_max": None, "time_len": None,
        "lat": None, "lon": None, "lev": None, "ilev": None,
        "vars": []
    }
    try:
        with xr.open_dataset(nc_path, decode_times=False, chunks={}) as ds:
            # time
            if "time" in ds.dims or "time" in ds.variables:
                t = ds.get("time")
                if t is not None:
                    vals = t.values
                    if hasattr(vals, "size") and vals.size > 0:
                        info["time_len"] = int(vals.size)
                        info["time_min"] = float(np.nanmin(vals))
                        info["time_max"] = float(np.nanmax(vals))
            # grid
            info["lat"]  = int(ds.dims.get("lat", ds.sizes["lat"])) if "lat" in ds.sizes else None
            info["lon"]  = int(ds.dims.get("lon", ds.sizes["lon"])) if "lon" in ds.sizes else None
            info["lev"]  = int(ds.sizes.get("lev"))  if "lev"  in ds.sizes else None
            info["ilev"] = int(ds.sizes.get("ilev")) if "ilev" in ds.sizes else None

            # variables（过滤坐标与维度名，只保留常用物理量名）
            drop_names = {"time","lat","lon","lev","ilev","nbnd","date","datesec","time_bnds","gw",
                          "hyam","hybm","hyai","hybi","P0"}
            vars_all = [v for v in ds.variables if v not in drop_names]
            # 只给一个简明的“变量清单概览”（按字母序）
            info["vars"] = sorted(vars_all)[:30]  # 控制长度
    except Exception as e:
        info["error"] = str(e)
    return info

# ----------------------------
# 4) 将一类文件汇总为简洁描述
# ----------------------------
def summarize_bucket(bucket_name: str, files: List[Path]) -> str:
    lines: List[str] = []
    files = sorted(files)
    n = len(files)
    # 为便于理解：列出文件所在“根目录集合”和一个代表性“命名规律”
    dirs = sorted({str(f.parent) for f in files})
    sample = files[0] if n > 0 else None

    # 取 2 个样本做 header 读取，合并范围/变量
    samples = [files[0]] if n >= 1 else []
    if n >= 2:
        samples.append(files[-1])

    time_min, time_max = np.inf, -np.inf
    lat = lon = lev = ilev = None
    vars_union: set = set()
    time_lens: Counter = Counter()

    for s in samples:
        meta = safe_read_meta(s)
        if meta.get("time_min") is not None:
            time_min = min(time_min, meta["time_min"])
        if meta.get("time_max") is not None:
            time_max = max(time_max, meta["time_max"])
        if meta.get("time_len") is not None:
            time_lens.update([meta["time_len"]])
        # 优先保留“更完整”的网格信息
        lat  = lat  or meta.get("lat")
        lon  = lon  or meta.get("lon")
        lev  = lev  or meta.get("lev")
        ilev = ilev or meta.get("ilev")
        vars_union.update(meta.get("vars", []))

    # 命名规律（由正则直接给出）
    pattern_text = PATTERNS.get(bucket_name, re.compile("")).pattern

    lines.append(f"[GROUP] {bucket_name}")
    lines.append(f"  - count        : {n}")
    if dirs:
        lines.append(f"  - dirs         :")
        for d in dirs[:5]:
            lines.append(f"      * {d}")
        if len(dirs) > 5:
            lines.append(f"      * ... ({len(dirs)-5} more)")
    if sample:
        lines.append(f"  - example      : {sample}")
    lines.append(f"  - naming.regex : {pattern_text}")

    # 时间范围与分辨率（抽样估计）
    if time_min != np.inf and time_max != -np.inf:
        lines.append(f"  - time.cover   : t0={time_min} ; t1={time_max} (units from file)")
    if time_lens:
        pretty = ", ".join([f"{k}:{v}" for k, v in sorted(time_lens.items())])
        lines.append(f"  - time_len.dist: {pretty}")
    if any([lat, lon, lev, ilev]):
        lines.append(f"  - grid         : lat={lat}, lon={lon}, lev={lev}, ilev={ilev}")

    if vars_union:
        # 仅概括性展示常见变量关键字（进一步压缩：优先展示你关心的）
        # 常见大气变量优先排序
        priority = ["U","V","T","Z3","O3","OMEGA","Q","PS","PSL","TREFHT",
                    "QRL","QRS","CLDTOT","SWCF","ICEFRAC","PHIS","PRECL","PRECC"]
        ordered = [v for v in priority if v in vars_union]
        the_rest = sorted([v for v in vars_union if v not in priority])
        brief = ordered + the_rest[:20]
        lines.append(f"  - vars.sampled : {', '.join(brief)}")

    return "\n".join(lines)

# ----------------------------
# 5) 主流程：扫描 -> 汇总 -> 输出
# ----------------------------
def scan_and_write(root: Path, out_txt: Path):
    buckets = collect_files(root)
    # 将我们关心的类目优先输出，最后输出 OTHER
    order = [k for k in PATTERNS.keys() if k in buckets] + (["OTHER"] if "OTHER" in buckets else [])
    out_lines: List[str] = []
    out_lines.append(f"# summary for: {root}")
    out_lines.append("# note: classes & regex are curated from your filetree to avoid missing any subtype")
    out_lines.append("")
    for key in order:
        out_lines.append(summarize_bucket(key, buckets[key]))
        out_lines.append("")

    out_txt.parent.mkdir(parents=True, exist_ok=True)
    with open(out_txt, "w", encoding="utf-8") as f:
        f.write("\n".join(out_lines))

# 运行
scan_and_write(ROOT, OUT)
print(f"Done. Wrote: {OUT}")
